# 1. Análisis Exploratorio de Datos (EDA) y Preprocesamiento

# Importacion y carga #

Primero vamos a cargar el dataset completo para realizar la limpieza sobre la población total antes de extraer ninguna muestra, garantizando así que los datos con los que entrenaremos los modelos sean íntegros.

In [1]:
# Importamos las librerías clásicas para análisis de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuramos el estilo visual de las gráficas
# sns.set_theme(style="whitegrid")

# Cargamos los datos completos
df = pd.read_csv('CVD_cleaned.csv')

print(f"Tamaño del dataset original: {df.shape}")
df.head(10)

Tamaño del dataset original: (308854, 19)


,General_Health,Checkup,Exercise,Heart_Disease,Skin_Cancer,Other_Cancer,Depression,Diabetes,Arthritis,Sex,Age_Category,Height_(cm),Weight_(kg),BMI,Smoking_History,Alcohol_Consumption,Fruit_Consumption,Green_Vegetables_Consumption,FriedPotato_Consumption
0,Poor,Within the past 2 years,No,No,No,No,No,No,Yes,Female,70-74,150.0,32.66,14.54,Yes,0.0,30.0,16.0,12.0
1,Very Good,Within the past year,No,Yes,No,No,No,Yes,No,Female,70-74,165.0,77.11,28.29,No,0.0,30.0,0.0,4.0
2,Very Good,Within the past year,Yes,No,No,No,No,Yes,No,Female,60-64,163.0,88.45,33.47,No,4.0,12.0,3.0,16.0
3,Poor,Within the past year,Yes,Yes,No,No,No,Yes,No,Male,75-79,180.0,93.44,28.73,No,0.0,30.0,30.0,8.0
4,Good,Within the past year,No,No,No,No,No,No,No,Male,80+,191.0,88.45,24.37,Yes,0.0,8.0,4.0,0.0
5,Good,Within the past year,No,No,No,No,Yes,No,Yes,Male,60-64,183.0,154.22,46.11,No,0.0,12.0,12.0,12.0
6,Fair,Within the past year,Yes,Yes,No,No,No,No,Yes,Male,60-64,175.0,69.85,22.74,Yes,0.0,16.0,8.0,0.0
7,Good,Within the past year,Yes,No,No,No,No,No,Yes,Female,65-69,165.0,108.86,39.94,Yes,3.0,30.0,8.0,8.0
8,Fair,Within the past year,No,No,No,No,Yes,No,No,Female,65-69,163.0,72.57,27.46,Yes,0.0,12.0,12.0,4.0
9,Fair,Within the past year,No,No,No,No,No,Yes,Yes,Female,70-74,163.0,91.63,34.67,No,0.0,12.0,12.0,1.0


# Limpieza de datos 

## 1.1 Identificación de Valores Nulos (NaN)
Comprovamos si existen campos vacíos en nuestro dataset que puedan hacer fallar a los modelos de Machine Learning.

In [2]:
# Contamos los nulos por cada columna
print("--- Valores nulos por columna ---")
print(df.isnull().sum())

# Guardamos el total de nulos para el resumen final
nulos_totales = df.isnull().sum().sum()
print(f"\nTotal absoluto de valores nulos en el dataset: {nulos_totales}")

--- Valores nulos por columna ---
General_Health                  0
Checkup                         0
Exercise                        0
Heart_Disease                   0
Skin_Cancer                     0
Other_Cancer                    0
Depression                      0
Diabetes                        0
Arthritis                       0
Sex                             0
Age_Category                    0
Height_(cm)                     0
Weight_(kg)                     0
BMI                             0
Smoking_History                 0
Alcohol_Consumption             0
Fruit_Consumption               0
Green_Vegetables_Consumption    0
FriedPotato_Consumption         0
dtype: int64

Total absoluto de valores nulos en el dataset: 0


## 1.2 Identificación y Eliminación de Filas Duplicadas
Vamos a buscar pacientes que tengan exactamente los mismos datos en todas las columnas. Esto suele ser un error en la recolección de datos y puede sesgar el entrenamiento del modelo.

In [3]:
# Guardamos el tamaño original para comparar después
filas_antes = len(df)

# 1. CREAMOS EL CLON EXACTO AQUÍ
df_clean = df.copy()

# 2. Contamos los duplicados sobre df_clean
duplicados = df_clean.duplicated().sum()
print(f"Se han detectado {duplicados} filas exactamente iguales en la población.")

# 3. Eliminamos los duplicados SOLO en df_clean
if duplicados > 0:
    df_clean.drop_duplicates(inplace=True)
    print(f"✅ ACCIÓN: Se han borrado {duplicados} líneas del dataset limpio (df_clean).")
else:
    print("✅ ACCIÓN: No se ha borrado ninguna línea porque no había duplicados.")

Se han detectado 80 filas exactamente iguales en la población.
✅ ACCIÓN: Se han borrado 80 líneas del dataset limpio (df_clean).


## 1.3 Diccionario de Datos Estructural
Generamos una tabla resumen con la anatomía de cada columna: el tipo de variable, la cantidad de respuestas diferentes, si contiene valores nulos, el rango y las categorías de texto. Esto nos permite comprobar la integridad estructural antes de hacer cálculos.

In [4]:
# Creamos la lista para el diccionario estructural
diccionario_datos = []

# CAMBIO: Usamos df_clean en lugar de df
for col in df_clean.columns:
    tipo_dato = str(df_clean[col].dtype)
    num_unicos = df_clean[col].nunique()
    nulos = df_clean[col].isnull().sum() 
    
    if tipo_dato == 'object':
        rango = "-" 
        respuestas = ", ".join([str(val) for val in df_clean[col].unique()]) 
    else:
        rango = f"{df_clean[col].min()} a {df_clean[col].max()}"
        respuestas = "-"
        
    diccionario_datos.append({
        'Columna': col,
        'Tipo de dato': tipo_dato,
        'Nulos': nulos,
        'Valores únicos': num_unicos,
        'Rango': rango,
        'Respuestas (Texto)': respuestas
    })

df_diccionario = pd.DataFrame(diccionario_datos)
df_diccionario

,Columna,Tipo de dato,Nulos,Valores únicos,Rango,Respuestas (Texto)
0,General_Health,object,0,5,-,"Poor, Very Good, Good, Fair, Excellent"
1,Checkup,object,0,5,-,"Within the past 2 years, Within the past year,..."
2,Exercise,object,0,2,-,"No, Yes"
3,Heart_Disease,object,0,2,-,"No, Yes"
4,Skin_Cancer,object,0,2,-,"No, Yes"
5,Other_Cancer,object,0,2,-,"No, Yes"
6,Depression,object,0,2,-,"No, Yes"
7,Diabetes,object,0,4,-,"No, Yes, No, pre-diabetes or borderline diabet..."
8,Arthritis,object,0,2,-,"Yes, No"
9,Sex,object,0,2,-,"Female, Male"


## 1.4 Análisis Estadístico Descriptivo (Solo variables numéricas)
Una vez verificada la estructura, profundizamos en el análisis matemático exclusivamente de las variables numéricas (como el peso, la altura o el consumo de alimentos). Estudiaremos las medidas de tendencia central (media, mediana, moda) y las medidas de dispersión (varianza y desviación típica) para entender la distribución real de los pacientes.

In [5]:
# CAMBIO: Usamos df_clean
cols_numericas = df_clean.select_dtypes(include=['int64', 'float64']).columns

estadisticas = []

for col in cols_numericas:
    estadisticas.append({
        'Variable Numérica': col,
        'Rango': f"{df_clean[col].min()} a {df_clean[col].max()}",
        '% Nulos': f"{round((df_clean[col].isnull().sum() / len(df_clean)) * 100, 2)}%",
        'Media': round(df_clean[col].mean(), 2),
        'Mediana': round(df_clean[col].median(), 2),
        'Moda': round(df_clean[col].mode()[0], 2),
        'Varianza': round(df_clean[col].var(), 2),
        'Desv. Típica (Std)': round(df_clean[col].std(), 2)
    })

df_estadisticas = pd.DataFrame(estadisticas)
df_estadisticas

,Variable Numérica,Rango,% Nulos,Media,Mediana,Moda,Varianza,Desv. Típica (Std)
0,Height_(cm),91.0 a 241.0,0.0%,170.62,170.00,168.00,113.60,10.66
1,Weight_(kg),24.95 a 293.02,0.0%,83.59,81.65,90.72,455.59,21.34
2,BMI,12.02 a 99.33,0.0%,28.63,27.44,26.63,42.55,6.52
3,Alcohol_Consumption,0.0 a 30.0,0.0%,5.10,1.00,0.00,67.25,8.20
4,Fruit_Consumption,0.0 a 120.0,0.0%,29.83,30.00,30.00,618.91,24.88
5,Green_Vegetables_Consumption,0.0 a 128.0,0.0%,15.11,12.00,30.00,222.81,14.93
6,FriedPotato_Consumption,0.0 a 128.0,0.0%,6.30,4.00,4.00,73.68,8.58


## 1.4 Resumen del Proceso de Limpieza
Tras analizar la población total de pacientes, este es el balance final de la calidad de nuestros datos antes de extraer la muestra de entrenamiento.

In [6]:
# CAMBIO: Obtenemos el tamaño final de df_clean
filas_despues = len(df_clean)

print("========== RESUMEN DE LIMPIEZA ==========")
print(f"-> Nulos encontrados y tratados: {nulos_totales}")
print(f"-> Filas duplicadas eliminadas:  {duplicados}")
print("-" * 41)
print(f"-> Tamaño inicial de la población (df): {filas_antes} filas.")
print(f"-> TAMAÑO FINAL POBLACIÓN LIMPIA (df_clean):  {filas_despues} filas.")
print(f"-> Total de características (columnas): {df_clean.shape[1]}")
print(f"Tamaño del dataset original: {df.shape}")
print(f"Tamaño del dataset limpio: {df_clean.shape}")
print("=========================================")

========== RESUMEN DE LIMPIEZA ==========
-> Nulos encontrados y tratados: 0
-> Filas duplicadas eliminadas:  80
-----------------------------------------
-> Tamaño inicial de la población (df): 308854 filas.
-> TAMAÑO FINAL POBLACIÓN LIMPIA (df_clean):  308774 filas.
-> Total de características (columnas): 19
Tamaño del dataset original: (308854, 19)
Tamaño del dataset limpio: (308774, 19)
